In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import datetime
import csv

# Configuration
RESULTS_DIR = Path("../results")


In [ ]:
# Path(Path(RESULTS_DIR.parent) / ".." / "measurement")
Path(Path(RESULTS_DIR.parent)).absolute()

In [ ]:
import time as t
import datetime as dt
dt.datetime.now().strftime("%Y-%m-%d %H%M%S_%f")

In [ ]:
t.strftime("%Y%m%d-%H%M%S")

In [ ]:
# Define time range for concatenating measurements
start_str ="2026-02-06 01:01:09"  # inclusive start
end_str = "2026-02-06 06:10:51"



type_meas = "full_swing" # full_swing

psd_analysis = True
plot_data = True    

modulation_frequency = 100e3
acquisition_frequency = 1e6
data_length = 1e4
data_time_span = 1e-3

time_for_quarter_period = 1/modulation_frequency/4
data_points_for_quarter_period = int(time_for_quarter_period*acquisition_frequency)


In [ ]:
%matplotlib widget

In [ ]:
start_dt = datetime.datetime.strptime(start_str, "%Y-%m-%d %H:%M:%S") - datetime.timedelta(seconds=1)
end_dt = datetime.datetime.strptime(end_str, "%Y-%m-%d %H:%M:%S") + datetime.timedelta(seconds=1)
if end_dt <= start_dt:
    raise ValueError("End datetime must be after start datetime")

# Find all files in the specified time range



In [ ]:
def files_list(type_meas, exclude_set = None, start_dt = start_dt, end_dt = end_dt):
    files_in_range = []
    
    # Handle exclude_set: can be None, a list, or a filename (string/Path)
    if exclude_set is None:
        exclude_set = []
    elif isinstance(exclude_set, (str, Path)):
        # If it's a filename, read it and create a list
        exclude_file_path = Path(exclude_set)
        
        # Try to find the file: first as-is, then relative to current directory, then relative to measurement directory
        if not exclude_file_path.is_absolute():
            if not exclude_file_path.exists():
                # Try relative to current directory
                exclude_file_path = Path.cwd() / exclude_file_path
            if not exclude_file_path.exists():
                # Try relative to measurement directory (where RESULTS_DIR is)
                exclude_file_path = RESULTS_DIR.parent / ".." / ".." / exclude_file_path
                print("use relative path")
        
        try:
            with open(exclude_file_path, 'r', encoding='utf-8') as f:
                exclude_set = [line.strip() for line in f if line.strip()]  # Read lines and strip whitespace
            print(f"Loaded {len(exclude_set)} files from exclude list: {exclude_file_path}")
        except FileNotFoundError:
            print(f"Warning: Exclude file not found: {exclude_file_path}. Using empty exclude list.")
            exclude_set = []
        except Exception as e:
            print(f"Warning: Error reading exclude file {exclude_file_path}: {e}. Using empty exclude list.")
            exclude_set = []
    
    current_date = start_dt.date()
    while current_date <= end_dt.date():
        print(current_date)
        for file in sorted((RESULTS_DIR  / current_date.strftime("%Y") / current_date.strftime("%m") / current_date.strftime("%d")).glob(f"{type_meas}_*.json")):
            # Skip if file is in exclusion list
            if file.name in exclude_set:
                print(f"  Excluding: {file.name}")
                continue
            
            # print(file.name)
            try:
                stamp = file.stem.split("_")[-1]  # expects v_meas_YYYYMMDD-HHMMSS.json
                dt = datetime.datetime.strptime(stamp, "%Y%m%d-%H%M%S%f")
                # print(dt)
            except ValueError:
                continue  # skip files that don't match the timestamp pattern
            if start_dt <= dt <= end_dt:
                print(dt)
                files_in_range.append((dt, file))
        current_date += datetime.timedelta(days=1)
    
    return files_in_range

In [ ]:
# exclude_set = ["full_swing_20260106-000022.json","full_swing_20260106-000310.json","full_swing_20260106-005157.json", 'full_swing_20260106-014647.json', 'full_swing_20260106-021808.json', 'full_swing_20260106-021841.json', 'full_swing_20260106-025901.json',"full_swing_20260106-030327.json"]
exclude_set = []
files_in_range_full_swing = files_list("full_swing", exclude_set = exclude_set)


if not files_in_range_full_swing:
    print("No files found in the specified range.")
else:
    # Load and concatenate data from all files
    concat_time_full_swing = []
    concat_voltage_sin = []
    concat_voltage_cos = []
    concat_voltage_avg = []
    concat_cosine_normalised = []
    concat_voltage_normalised = []
    concat_phase = []
    concat_power = []
    concat_polarisation_measure = []
    concat_error = []
    base_dt = files_in_range_full_swing[0][0]  # anchor absolute time to the first capture
    lo_sin = []
    lo_cos = []
    los_sin_cos = []

    for dt, file in files_in_range_full_swing[::10]:
        with open(file, "r") as f:
            d = json.load(f)
        sampling_rate = d['sample_rate_hz']
        modulation_frequency = d['modulation_frequency_hz']
        time_for_quarter_period = 1/modulation_frequency/4
        data_points_for_quarter_period = int(time_for_quarter_period*sampling_rate)+1
        t = np.array(d["time_s"], dtype=float)
        v = np.array(d["voltage_data_v"], dtype=float)
        v_normalised = (v-np.mean(v))/(np.max(v)-np.min(v))
        cosine_filter = np.array(d["cosine_reference_v"], dtype=float)
        cosine_normalised = (cosine_filter-np.mean(cosine_filter))/(np.max(cosine_filter)-np.min(cosine_filter))
        sine_normalised = list(cosine_normalised[data_points_for_quarter_period+1:]) + list(cosine_normalised[:data_points_for_quarter_period+1])
        sine_normalised = np.array(sine_normalised)
        cosine_phase = sum(v_normalised*cosine_normalised)
        sine_phase = sum(v_normalised*sine_normalised)
        phase = np.arctan2(sine_phase, cosine_phase)*180/np.pi
        power = np.mean(v)
        polarisation_measure = (max(v) - power)/power
        error = (np.abs(cosine_normalised[-1] - cosine_normalised[0]) + np.abs(sine_normalised[-1] - sine_normalised[0]))/10
        sanity_lo_sin = sum(sine_normalised*sine_normalised)
        sanity_lo_cos = sum(cosine_normalised*cosine_normalised)
        sanity_los_sin_cos = sum(cosine_normalised*sine_normalised)
        if len(t) == 0 or len(v) == 0:
            continue

        # Offset this capture so its start reflects the true wall-clock interval
        offset = (dt - base_dt).total_seconds()
        print(dt - base_dt)
        concat_time_full_swing.append([(t + offset)[0]])
        # concat_voltage_sin.append([(v*sine_normalised).sum()/len(v)])
        # concat_voltage_cos.append([(v*cosine_filter).sum()/len(v)])
        # concat_voltage_avg.append([(v).sum()])
        # concat_cosine_normalised.append((cosine_normalised).tolist())
        # concat_voltage_normalised.append((v_normalised).tolist())
        concat_phase.append([phase])
        concat_power.append([power])
        concat_polarisation_measure.append([polarisation_measure])
        concat_error.append([error])
        lo_sin.append([sanity_lo_sin])
        lo_cos.append([sanity_lo_cos])
        los_sin_cos.append([sanity_los_sin_cos])
    if not concat_time_full_swing:
        print("All files in range were empty after parsing.")
    else:
        concat_time_full_swing = np.concatenate(concat_time_full_swing)
        # concat_voltage_sin = np.concatenate(concat_voltage_sin)
        # concat_voltage_cos = np.concatenate(concat_voltage_cos)
        # concat_voltage_avg = np.concatenate(concat_voltage_avg)
        # concat_cosine_normalised = np.concatenate(concat_cosine_normalised)
        # concat_voltage_normalised = np.concatenate(concat_voltage_normalised)
        concat_phase = np.concatenate(concat_phase)
        concat_power = np.concatenate(concat_power)
        concat_polarisation_measure = np.concatenate(concat_polarisation_measure)
        concat_error = np.concatenate(concat_error)
        lo_sin = np.concatenate(lo_sin)
        lo_cos = np.concatenate(lo_cos)
        los_sin_cos = np.concatenate(los_sin_cos)
        print(f"Loaded {len(files_in_range_full_swing)} files")
        print(f"Total data points: {len(concat_time_full_swing)}")
        print(f"Time range: {base_dt} to {base_dt + datetime.timedelta(seconds=concat_time_full_swing[-1])}")



In [ ]:
from datetime import timedelta
concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]
# concat_datetime[-1]

In [ ]:
def jump_removal(values, max_discontinuity):
    """
    Remove large step jumps from a 1D sequence.
    If abs(values[i] - values[i-1]) > max_discontinuity, the function
    shifts all subsequent values so that the jump at i becomes zero.
    """
    # if not values:
    #     return []
    corrected = list(values)  # keep input unchanged
    for i in range(1, len(corrected)):
        delta = corrected[i] - corrected[i - 1]
        if abs(delta) > max_discontinuity:
            # Remove the whole jump so corrected[i] == corrected[i-1]
            for j in range(i, len(corrected)):
                corrected[j] -= delta
    return np.array(corrected)

In [ ]:
concat_power_normalised = (concat_power)/max(concat_power)
concat_polarisation_normalised  = (concat_polarisation_measure)/max(concat_polarisation_measure)
concat_polarization_phase = np.arccos(concat_polarisation_normalised)*180/np.pi
concat_phase_adjusted_winding = np.array([phase + 360 if phase < 0 else phase for phase in concat_phase])
concat_phase_unwrap = np.unwrap(concat_phase*2*np.pi/180)*180/(2*np.pi)
concat_phase_smooth = jump_removal(concat_phase_unwrap, 20)

In [ ]:
# Plot of LO Sanity
# Plot concat_power_normalised, concat_polarization_phase, and concat_phase (with error bars)
from datetime import timedelta
import matplotlib.dates as mdates
from matplotlib.dates import SecondLocator, MinuteLocator, HourLocator

if 'concat_time_full_swing' in locals() and len(concat_time_full_swing) > 0:
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]

    fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # Subplot 1: concat_power_normalised
    axs[0].plot(concat_datetime, lo_sin, linewidth=1, color='tab:blue')
    axs[0].set_ylabel("Sin^2", fontsize=12)
    axs[0].set_title(f"LO Sanity; Data: {start_str} to {end_str} ({len(files_in_range_full_swing)} files)", fontsize=14)
    axs[0].grid(True, alpha=0.3)

    # Subplot 2: concat_polarization_phase
    axs[1].plot(concat_datetime, lo_cos, linewidth=1, color='tab:orange')
    axs[1].set_ylabel("Cos^2", fontsize=12)
    axs[1].grid(True, alpha=0.3)

    # Subplot 3: concat_phase with error bars
    axs[2].plot(concat_datetime, los_sin_cos/(lo_sin),
                    linewidth=1, color='tab:green', label='phase')
    # axs[2].errorbar(concat_datetime, np.array(concat_phase_adjusted_winding),
    #                 yerr=np.array(concat_error),
    #                 fmt='-', linewidth=1, elinewidth=0.5, capsize=2,
    #                 color='tab:green', ecolor='tab:green', alpha=0.8, label='phase +/- error')
    axs[2].set_ylabel("Sin*Cos/Cos^2", fontsize=12)
    axs[2].set_xlabel("Time", fontsize=12)
    axs[2].legend(loc='upper right')
    axs[2].grid(True, alpha=0.3)

    # Format x-axis
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs[2].xaxis.set_major_locator(HourLocator(interval=1))
    plt.setp(axs[2].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    plt.savefig(f"combined_plot_{start_str}_{end_str}.png")
    plt.show()
else:
    print("No data loaded. Please run the data loading cell first.")

In [ ]:
# Plot concat_power_normalised, concat_polarization_phase, and concat_phase (with error bars)
from datetime import timedelta
import matplotlib.dates as mdates
from matplotlib.dates import SecondLocator, MinuteLocator, HourLocator

if 'concat_time_full_swing' in locals() and len(concat_time_full_swing) > 0:
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]

    fig, axs = plt.subplots(4, 1, figsize=(12, 9), sharex=True)

    # Subplot 1: concat_power_normalised
    axs[0].plot(concat_datetime, concat_power_normalised, linewidth=1, color='tab:blue')
    axs[0].set_ylabel("Normalised Power", fontsize=12)
    axs[0].set_title(f"Data: {start_str} to {end_str} ({len(files_in_range_full_swing)} files)", fontsize=14)
    axs[0].grid(True, alpha=0.3)

    # Subplot 2: concat_polarization_phase
    axs[1].plot(concat_datetime, concat_polarization_phase, linewidth=1, color='tab:orange')
    axs[1].set_ylabel("Polarization Phase (deg)", fontsize=12)
    axs[1].grid(True, alpha=0.3)

    # Subplot 3: concat_phase with error bars
    axs[2].plot(concat_datetime, np.array(concat_phase),
                    linewidth=1, color='tab:green', label='phase')
    # axs[2].errorbar(concat_datetime, np.array(concat_phase_adjusted_winding),
    #                 yerr=np.array(concat_error),
    #                 fmt='-', linewidth=1, elinewidth=0.5, capsize=2,
    #                 color='tab:green', ecolor='tab:green', alpha=0.8, label='phase +/- error')
    axs[2].set_ylabel("Phase (deg)", fontsize=12)
    axs[2].set_xlabel("Time", fontsize=12)
    axs[2].legend(loc='upper right')
    axs[2].grid(True, alpha=0.3)

    # Format x-axis
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs[2].xaxis.set_major_locator(HourLocator(interval=1))
    axs[3].plot(concat_datetime, concat_phase_unwrap, linewidth=1, color='tab:red', label='adjusting for phase winding')
    axs[3].plot(concat_datetime, concat_phase_smooth, linewidth=1, color='tab:blue', label='adjusting for phase discontinuity from clock reset')
    axs[3].legend(loc='upper right')
    axs[3].grid(True, alpha=0.3)
    plt.setp(axs[2].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    # plt.savefig(f"combined_plot_{start_str}_{end_str}.png")
    plt.show()
else:
    print("No data loaded. Please run the data loading cell first.")


In [ ]:
# Plot concat_power_normalised, concat_polarization_phase, and concat_phase (with error bars)
from datetime import timedelta
import matplotlib.dates as mdates
from matplotlib.dates import SecondLocator, MinuteLocator, HourLocator

if 'concat_time_full_swing' in locals() and len(concat_time_full_swing) > 0:
    concat_datetime = [base_dt + timedelta(seconds=float(t)) for t in concat_time_full_swing]

    fig, axs = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    # Subplot 1: concat_power_normalised
    axs[0].plot(concat_datetime, concat_power_normalised, linewidth=1, color='tab:blue')
    axs[0].set_ylabel("Normalised Power", fontsize=12)
    axs[0].set_title(f"Data: {start_str} to {end_str} ({len(files_in_range_full_swing)} files)", fontsize=14)
    axs[0].grid(True, alpha=0.3)

    # Subplot 2: concat_polarization_phase
    axs[1].plot(concat_datetime, concat_polarization_phase, linewidth=1, color='tab:orange')
    axs[1].set_ylabel("Polarization Phase (deg)", fontsize=12)
    axs[1].grid(True, alpha=0.3)

    # Subplot 3: concat_phase with error bars
    axs[2].plot(concat_datetime, np.array(concat_phase_smooth) - (concat_phase_smooth[0]),
                    linewidth=1, color='tab:green', label='phase')
    # axs[2].errorbar(concat_datetime, np.array(concat_phase_adjusted_winding),
    #                 yerr=np.array(concat_error),
    #                 fmt='-', linewidth=1, elinewidth=0.5, capsize=2,
    #                 color='tab:green', ecolor='tab:green', alpha=0.8, label='phase +/- error')
    axs[2].set_ylabel("Phase (deg)", fontsize=12)
    axs[2].set_xlabel("Time", fontsize=12)
    axs[2].legend(loc='upper right')
    axs[2].grid(True, alpha=0.3)

    # Format x-axis
    axs[2].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axs[2].xaxis.set_major_locator(HourLocator(interval=1))

    plt.setp(axs[2].xaxis.get_majorticklabels(), rotation=45)

    plt.tight_layout()
    # plt.savefig(f"combined_plot_{start_str}_{end_str}.png")
    plt.show()
else:
    print("No data loaded. Please run the data loading cell first.")


In [ ]:
# Power spectral density (PSD) of phase
if 'concat_time_full_swing' in locals() and 'concat_phase_smooth' in locals() and len(concat_time_full_swing) > 2 and psd_analysis:
    # Use capture times as the time axis (seconds from base_dt)
    t_phase = np.asarray(concat_time_full_swing, dtype=float)

    # concat_phase is stored in degrees; unwrap in radians to avoid 360 deg jumps
    phase_deg = np.asarray(concat_phase_smooth, dtype=float)
    phase_rad = np.deg2rad(phase_deg)

    # Build a uniform time grid for FFT-based PSD (robust to timing jitter)
    dt_median = np.median(np.diff(t_phase))
    if dt_median <= 0:
        raise ValueError("Non-positive time step detected in concat_time_full_swing")

    fs = 1.0 / dt_median
    t_uniform = np.arange(t_phase[0], t_phase[-1] + 0.5 * dt_median, dt_median)



    
    # phase_uniform = np.interp(t_uniform, t_phase, phase_rad_unwrapped) Skipping this step

    # Detrend (remove DC) before PSD
    phase_zero_mean = phase_rad - np.mean(phase_rad)
    phase_hamming = phase_zero_mean * np.hamming(len(phase_zero_mean))
    n = len(phase_hamming)

    # One-sided PSD via rFFT; units: rad^2/Hz
    fft_vals = np.fft.rfft(phase_hamming)
    freqs = np.fft.rfftfreq(n, d=dt_median)
    psd = (np.abs(fft_vals) ** 2) / (fs * n)
    if n > 1:
        psd[1:-1] *= 2.0

    # Plot both views in one figure: frequency and time scale
    valid = freqs > 0
    time_scales = 1.0 / freqs[valid]
    sort_idx = np.argsort(time_scales)

    fig, (ax_freq, ax_time) = plt.subplots(2, 1, figsize=(10, 9))

    ax_freq.loglog(freqs[valid], psd[valid], linewidth=1.2)
    ax_freq.set_xlabel('Frequency (Hz)', fontsize=12)
    ax_freq.set_ylabel('PSD of phase (rad^2/Hz)', fontsize=12)
    ax_freq.set_title(f'Phase PSD vs Frequency: {start_str} to {end_str}', fontsize=14)
    ax_freq.grid(True, which='both', alpha=0.3)

    ax_time.loglog(time_scales[sort_idx], psd[valid][sort_idx], linewidth=1.2)
    ax_time.set_xlabel('Time scale (s)', fontsize=12)
    ax_time.set_ylabel('PSD of phase (rad^2/Hz)', fontsize=12)
    ax_time.set_title(f'Phase PSD vs Time Scale: {start_str} to {end_str}', fontsize=14)
    ax_time.grid(True, which='both', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Estimated sampling rate from timestamps: {fs:.3f} Hz")
    print(f"Frequency span: {freqs[1]:.6f} Hz to {freqs[-1]:.3f} Hz")
else:
    print("No phase data loaded. Please run the data loading cell first.")
